# Chapter 18: Semi-Supervised Learning

Semi-supervised learning uses a small amount of labeled data and a large amount of unlabeled data.

### High-Level Intuition & Analogy
Labeling data is expensive. If we have a few labeled points and many unlabeled points, we can construct a graph connecting similar points and let the labels "spread" across the network like dye in water (**Label Propagation**).

### Core Concepts & Step-by-Step Execution
1. **Graph Construction**: Connect data points with edges weighted by similarity.
2. **Label Diffusion**: Let labels flow along edges to unlabeled neighbors.
3. **Normalization**: Normalize probabilities at each step and clamp labeled points to their true values.

### Mathematical Foundations (Simplified)
* **Transition Probability**: $P_{i,j} = \frac{W_{i,j}}{\sum_k W_{i,k}}$, where $W$ is the similarity matrix. Unlabeled labels are iteratively updated by $Y \leftarrow P \cdot Y$ until convergence.

### Pros and Cons
* **Pros**: Reduces annotation costs, and leverages unlabeled datasets.
* **Cons**: If initial labels are noisy, the errors will propagate throughout the entire graph.

### Pro-Tips & Gotchas
* **Out-of-Distribution Nodes**: Unlabeled points that lie far away from clusters can end up with random labels. Use clean thresholds to flag and ignore predictions for isolated nodes.

### Programming Guide & Key Functions
Here is a comprehensive label propagation script:

```python
from sklearn.semi_supervised import LabelPropagation, LabelSpreading
import numpy as np

X = np.array([[1, 2], [1.5, 1.8], [5, 8], [8, 8], [1, 0.6], [9, 11]])
y_mixed = np.array([0, 1, -1, -1, 0, -1])  # -1 represents unlabeled samples

# --- Label Propagation (basic) ---
lp = LabelPropagation(kernel='knn', n_neighbors=2)
lp.fit(X, y_mixed)
predicted_labels = lp.transduction_  # Access predictions for all samples
probs = lp.label_distributions_  # Class probability distribution per sample

# --- Label Spreading (adds regularization to handle noisy edges) ---
ls = LabelSpreading(kernel='rbf', alpha=0.2)  # alpha controls clamping strength
ls.fit(X, y_mixed)
```

In [ ]:
from sklearn.semi_supervised import LabelPropagation
from sklearn.datasets import make_classification
import numpy as np

# Make data where -1 denotes unlabeled points
X, y = make_classification(n_samples=100, n_features=4, n_classes=2, random_state=42)
y_masked = y.copy()
y_masked[80:] = -1  # Mask 20% labels as unlabeled

lp = LabelPropagation().fit(X, y_masked)
print("Transduced Labels for unlabeled points:", lp.transduction_[80:])

---
## Exercises

Now it is your turn! Run the verification code cell after each exercise to check your answers.

### Exercise 1: Import Label Propagation (Easy)
Import `LabelPropagation` from `sklearn.semi_supervised` and instantiate it. Store it in a variable named `lp_easy`.

In [ ]:
# TODO: Import and instantiate LabelPropagation
lp_easy = ____
print(lp_easy)

In [ ]:
from learntools import ch18

ch18.step_1.check(lp_easy)
# ch18.step_1.hint()# ch18.step_1.solution()

### Exercise 2: Count Unlabeled Samples (Easy)
In semi-supervised learning, unlabeled samples are represented by a label of `-1`. Given labels `y_masked`, count the number of unlabeled samples. Store it in `unlabeled_count`.

In [ ]:
y_masked = np.array([0, 1, -1, 0, -1, 1, -1])
# TODO: Count how many labels are equal to -1
unlabeled_count = ____
print(unlabeled_count)

In [ ]:
ch18.step_2.check(unlabeled_count)
# ch18.step_2.hint()# ch18.step_2.solution()

### Exercise 3: Import Label Spreading (Easy)
Import `LabelSpreading` from `sklearn.semi_supervised` and instantiate it. Store it in a variable named `ls_easy`.

In [ ]:
# TODO: Import and instantiate LabelSpreading
ls_easy = ____
print(ls_easy)

In [ ]:
ch18.step_3.check(ls_easy)
# ch18.step_3.hint()# ch18.step_3.solution()

### Exercise 4: LabelSpreading model (Easy)

Import and instantiate a `LabelSpreading` model with a `'knn'` kernel and `n_neighbors=5` and store it in `ls_model`.

In [ ]:
from sklearn.semi_supervised import LabelSpreading

# TODO: Instantiate LabelSpreading

ls_model = ____

In [ ]:
ch18.step_7.check(ls_model)

# ch18.step_7.hint()# ch18.step_7.solution()

### Exercise 5: Mask unlabeled elements (Easy/Medium)

Create a copy of label Series `y_raw` and replace elements where value is `'unknown'` with label `-1`, storing the semi-supervised labels array in `y_semi`.

In [ ]:
y_raw = pd.Series([1, 0, "unknown", 1, "unknown"])

# TODO: Mask unknown labels to -1

y_semi = ____

print(y_semi)

In [ ]:
ch18.step_8.check(y_semi)

# ch18.step_8.hint()# ch18.step_8.solution()

### Exercise 6: Graph-based Label Propagation (Medium)
Train a `LabelPropagation` model on the dataset where **90%** of labels are masked as unlabeled (`-1`).
1. Use `kernel='knn'` and `n_neighbors=5`.
2. Fit the model on training features `X_train` and labels `y_train` (containing `-1` masks).
3. Compute the accuracy score of the propagated labels on the test mask items. Store your fitted model object in `lp_model` and the accuracy score in `acc`.

In [ ]:
from sklearn.semi_supervised import LabelPropagation
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X, y = make_classification(n_samples=500, n_features=6, n_classes=2, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Mask 90% of training labels
np.random.seed(42)
mask = np.random.rand(len(y_train)) < 0.9
y_train_masked = y_train.copy()
y_train_masked[mask] = -1

# TODO: Train LabelPropagation model and evaluate accuracy on test set
lp_model = ____
acc = ____

print(f"Transduction Accuracy: {acc:.4f}")

In [ ]:
ch18.step_4.check(lp_model, acc)
# ch18.step_4.hint()# ch18.step_4.solution()

### Exercise 7: Self-Training Pseudo-Labeling Loop (Medium)
Implement a single iteration of Pseudo-Labeling.
Write a function `pseudo_labeling(X, y_masked, threshold=0.9)` that:
1. Splits the training set into labeled indices (where `y_masked != -1`) and unlabeled indices (where `y_masked == -1`).
2. Trains a `LogisticRegression` classifier on the labeled data.
3. Predicts probabilities for the unlabeled data.
4. Identifies samples where the predicted probability of the predicted class is strictly greater than or equal to `threshold`.
5. Adds these highly confident samples and their predicted labels to the labeled dataset.
6. Retrains a final `LogisticRegression` classifier on the combined dataset and returns this fitted classifier object.

In [ ]:
from sklearn.linear_model import LogisticRegression


def pseudo_labeling(X, y_masked, threshold=0.9):
    # TODO: Implement pseudo-labeling iteration
    pass


# Test on a small set
X_mock = np.random.randn(10, 2)
y_mock = np.array([0, 1, 0, 1, 0, -1, -1, -1, -1, -1])
clf = pseudo_labeling(X_mock, y_mock)
print("Classifier trained successfully:", clf)

In [ ]:
ch18.step_5.check(pseudo_labeling)
# ch18.step_5.hint()# ch18.step_5.solution()

### Exercise 8: Label Spreading Kernel Hyperparameter Tuning (Hard)
Evaluate the impact of the Radial Basis Function (RBF) kernel `gamma` parameter on the `LabelSpreading` model.
The concentric moons dataset is non-linear.
Evaluate `gamma` values `[0.1, 1.0, 10.0, 100.0]`.
1. For each gamma, fit `LabelSpreading(kernel='rbf', gamma=gamma)` on the training set `(X_train, y_train_masked)`.
2. Compute the accuracy of the propagated labels against the actual labels of the test mask indices (where `y_train_masked == -1`).
3. Store the best gamma found in `best_gamma` and its corresponding accuracy in `best_acc`.

In [ ]:
from sklearn.semi_supervised import LabelSpreading
from sklearn.datasets import make_moons

X, y = make_moons(n_samples=500, noise=0.15, random_state=42)
y_masked = y.copy()
np.random.seed(42)
mask = np.random.rand(len(y)) < 0.8
y_masked[mask] = -1  # Mask 80% labels

# TODO: Loop over gammas, train LabelSpreading, find best_gamma and best_acc
best_gamma = 0.0
best_acc = 0.0

print(f"Best Gamma: {best_gamma} with Test Mask Accuracy: {best_acc:.4f}")

In [ ]:
ch18.step_6.check(best_gamma, best_acc)
# ch18.step_6.hint()# ch18.step_6.solution()